# Z3rno + OpenAI Agents Integration

This notebook demonstrates how to use Z3rno memory with OpenAI's function calling API to build agents that store and recall memories across conversations.

**Prerequisites:**
- A running z3rno-server instance (see [self-hosting docs](../self-hosting))
- An OpenAI API key
- Z3rno API key

In [ ]:
# Install dependencies
!pip install z3rno[openai] openai

## Setup

Configure your Z3rno and OpenAI credentials.

In [ ]:
import os
from openai import OpenAI

# Configuration
os.environ["OPENAI_API_KEY"] = "sk-your-openai-key-here"

Z3RNO_BASE_URL = "http://localhost:8000"
Z3RNO_API_KEY = "z3rno_sk_test_abc123def456"

client = OpenAI()

## Setting Up Memory Tools

Z3rno provides pre-built OpenAI function tool definitions that let your agent
store and recall memories. The `get_memory_tools()` function returns tool schemas
compatible with OpenAI's API.

In [ ]:
from z3rno.integrations.openai import get_memory_tools, handle_tool_call

# Get the Z3rno memory tool definitions for OpenAI function calling
memory_tools = get_memory_tools(
    base_url=Z3RNO_BASE_URL,
    api_key=Z3RNO_API_KEY,
    user_id="user_42",
)

# Inspect the available tools
for tool in memory_tools:
    print(f"Tool: {tool['function']['name']}")
    print(f"  Description: {tool['function']['description'][:80]}...")
    print()

## Multi-Turn Conversation with Memory

Here we build an agent loop where the assistant can store facts to Z3rno and
recall them later. The `handle_tool_call` utility processes tool calls and
returns the results.

In [ ]:
import json

SYSTEM_PROMPT = """You are a helpful assistant with persistent memory.
When the user shares important information (preferences, facts, context),
use the store_memory tool to save it for later.
When you need to recall past information, use the recall_memories tool.
Always check your memory before answering questions about the user."""

def run_agent(messages: list) -> str:
    """Run the agent loop, handling tool calls until a final response."""
    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=memory_tools,
            tool_choice="auto",
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message.model_dump())

        # If no tool calls, return the final response
        if not assistant_message.tool_calls:
            return assistant_message.content

        # Process each tool call
        for tool_call in assistant_message.tool_calls:
            result = handle_tool_call(
                tool_call=tool_call,
                base_url=Z3RNO_BASE_URL,
                api_key=Z3RNO_API_KEY,
                user_id="user_42",
            )
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            })

    return assistant_message.content

In [ ]:
# Start a conversation - share some preferences
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

# Turn 1: Share information
messages.append({"role": "user", "content": "I'm a vegetarian and I'm allergic to nuts. My favorite cuisine is Japanese."})
response = run_agent(messages)
print(f"Assistant: {response}")

In [ ]:
# Turn 2: Ask something that requires recalling stored memories
messages.append({"role": "user", "content": "Can you suggest a restaurant for me tonight?"})
response = run_agent(messages)
print(f"Assistant: {response}")

In [ ]:
# Turn 3: New session - memories persist!
# Simulate a completely new conversation (no prior messages in context)
new_messages = [{"role": "system", "content": SYSTEM_PROMPT}]
new_messages.append({"role": "user", "content": "What do you remember about my dietary restrictions?"})

response = run_agent(new_messages)
print(f"Assistant: {response}")
print("\n(The agent recalled memories from the previous conversation!)")

## Automatic Conversation Memory with Z3rnoConversationMemory

For a more hands-off approach, `Z3rnoConversationMemory` automatically stores
every message in a conversation without requiring explicit tool calls. This is
useful when you want full conversation logs persisted.

In [ ]:
from z3rno.integrations.openai import Z3rnoConversationMemory

# Initialize conversation memory
conv_memory = Z3rnoConversationMemory(
    base_url=Z3RNO_BASE_URL,
    api_key=Z3RNO_API_KEY,
    user_id="user_42",
    session_id="session_openai_demo",
)

# Add messages - these are automatically persisted to Z3rno
conv_memory.add_user_message("I need help planning a trip to Tokyo next month.")

response = client.chat.completions.create(
    model="gpt-4o",
    messages=conv_memory.get_messages(system_prompt="You are a helpful travel assistant."),
)

assistant_reply = response.choices[0].message.content
conv_memory.add_assistant_message(assistant_reply)

print(f"Assistant: {assistant_reply}")

In [ ]:
# Later - load the conversation from Z3rno
restored_memory = Z3rnoConversationMemory(
    base_url=Z3RNO_BASE_URL,
    api_key=Z3RNO_API_KEY,
    user_id="user_42",
    session_id="session_openai_demo",  # same session ID
)

# All messages are loaded from the server
messages = restored_memory.get_messages(system_prompt="You are a helpful travel assistant.")
print(f"Restored {len(messages)} messages from Z3rno")
for msg in messages:
    print(f"  [{msg['role']}]: {msg['content'][:60]}...")

## Summary

The Z3rno OpenAI integration provides:

- **Function tools** (`get_memory_tools`) — let the agent decide when to store/recall
- **Tool call handler** (`handle_tool_call`) — process memory tool calls with one function
- **Conversation memory** (`Z3rnoConversationMemory`) — automatic message persistence
- **Cross-session recall** — memories persist and are searchable across conversations

See the [Z3rno OpenAI docs](../integrations/openai) for full API reference.